# Multi-Task Gesture Recognition with Face and Pose Features

This notebook trains a multi-task model for gesture recognition that incorporates:
- Hand keypoints (42 features)
- Face features: head pose, eye gaze, mouth openness (10 features)
- Pose features: shoulder positions, torso orientation (8 features)
- Relative position features: hand-to-head, hand-to-shoulder distances (20 features)

Total: 80 features per frame

The model performs multi-task learning with:
1. Main task: Gesture classification
2. Auxiliary task: Hand region classification (face_side, chest, side, extended_forward)
3. Auxiliary task: Pointing direction
4. Auxiliary task: Grammatical categories

In [ ]:
import csv
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_SEED = 42

# Dataset Paths

In [ ]:
import os
import glob
import numpy as np

# Load all CSV files from Words-Dataset folder
csv_files = sorted(glob.glob('Words-Dataset/???.csv'))
data_frames = []
for csv_file in csv_files:
    data = np.loadtxt(csv_file, delimiter=',', dtype='float32')
    data_frames.append(data)
if data_frames:
    dataset = np.vstack(data_frames)
else:
    raise ValueError("No data found in Words-Dataset folder")

model_save_path = 'model/keypoint_classifier/multi_task_classifier.keras'

# Feature dimensions (using only hand features as available in current dataset)
HAND_FEATURES = 42  # 21 landmarks * 2 coordinates
FACE_FEATURES = 0   # Not available in current dataset
POSE_FEATURES = 0   # Not available in current dataset
RELATIVE_FEATURES = 0  # Not available in current dataset
TOTAL_FEATURES = HAND_FEATURES  # 42

# Multi-task outputs (will be set after loading data)
REGION_CLASSES = 4   # face_side, chest, side, extended_forward
DIRECTION_CLASSES = 8  # 8 pointing directions
GRAMMAR_CLASSES = 3   # semantic categories

# Load Dataset

In [ ]:
# Load main dataset
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, TOTAL_FEATURES + 1)))
y_gesture = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))
y_gesture -= 1  # Convert from 1-based to 0-based indexing for TensorFlow

# Set gesture classes dynamically
GESTURE_CLASSES = len(np.unique(y_gesture))  # Main gesture classes

print(f"Dataset shape: {X_dataset.shape}")
print(f"Gesture classes: {GESTURE_CLASSES}")

# Split dataset (single task learning with hand features only)
X_train, X_test, y_gesture_train, y_gesture_test = train_test_split(
    X_dataset, y_gesture, train_size=0.75, random_state=RANDOM_SEED, stratify=y_gesture)

# Build Multi-Task Model

In [ ]:
# Build Single-Task Model (using only hand features available in dataset)
model = tf.keras.models.Sequential([
    tf.keras.layers.Input((TOTAL_FEATURES, )),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(40, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(20, activation='relu'),
    tf.keras.layers.Dense(GESTURE_CLASSES, activation='softmax')
])

model.summary()

In [ ]:
# Model compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

# Train Model

In [ ]:
model.fit(
    X_train,
    y_gesture_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_gesture_test),
    callbacks=[cp_callback, es_callback]
)

In [ ]:
# Evaluate model
val_loss, val_acc = model.evaluate(X_test, y_gesture_test, batch_size=128)
print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}')

# Load saved model
model = tf.keras.models.load_model(model_save_path)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save TFLite model
tflite_save_path = 'model/keypoint_classifier/keypoint_classifier.tflite'
with open(tflite_save_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model saved to: {tflite_save_path}")